# Lab 08 — Geometric Shape Descriptors & Shape Diagrams

**Course:** Image Analysis and Processing — École des Mines de Saint-Étienne  
**Dataset:** Kimia-99 silhouette database (`apple`, `Bone`, `camel` — 20 shapes per class)  
**Topics covered:**
1. Geometric functionals from integral geometry and mathematical morphology:
   - **Crofton perimeter** $P_{\text{Crofton}}$ (4-orientation intercept estimation)
   - **Feret diameters** $d$ (minimum) and $D$ (maximum)
   - **Inscribed circle radius** $r$ (via chamfer distance transform)
2. Dimensionless shape descriptors: Elongation ($d/D$), Thinness ($2r/D$), Roundness ($4A / \pi D^2$), and Crofton Ratio ($P_{\text{Crofton}} / \pi D$)
3. Unsupervised K-Means clustering accuracy across three canonical shape diagrams

In [ ]:
from pathlib import Path
import numpy as np
import scipy.ndimage
from scipy.ndimage import rotate
import scipy.stats
import imageio.v3 as imageio
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
KIMIA_DIR = Path("../Images/images_Kimia")

## 1. Geometric Functionals

We implement the three core geometric measurements used to build dimensionless shape descriptors.

In [ ]:
def crofton_perimeter(binary_img: np.ndarray) -> float:
    """Estimate the perimeter using the 4-direction discrete Crofton formula."""
    intercepts = []
    kernel = np.array([[1, -1]])
    for i in range(4):
        rotated = rotate(binary_img.astype(float), 45 * i, reshape=True, order=0) > 0.5
        transitions = scipy.ndimage.convolve(rotated.astype(int), kernel)
        intercepts.append(np.sum(transitions > 0))

    return float(
        (np.pi / 4.0) * (intercepts[0] + intercepts[2] + (intercepts[1] + intercepts[3]) / np.sqrt(2))
    )


def feret_diameters(binary_img: np.ndarray):
    """Compute minimum (d) and maximum (D) Feret diameters across 30-degree steps."""
    d = float(np.max(binary_img.shape))
    D = 0.0
    img_float = binary_img.astype(float)

    for angle in np.arange(0, 180, 30):
        rotated = rotate(img_float, angle, reshape=True, order=0) > 0.5
        projection_width = float(np.sum(np.max(rotated, axis=0)))
        d = min(d, projection_width)
        D = max(D, projection_width)

    return d, D


def inscribed_radius(binary_img: np.ndarray) -> float:
    """Compute the maximum inscribed circle radius via chamfer distance transform."""
    dm = scipy.ndimage.distance_transform_cdt(binary_img > 0.5)
    return float(np.max(dm))

## 2. Feature Extraction Pipeline (Kimia Dataset)

For every silhouette in `apple`, `Bone`, and `camel`, we compute:
- **Elongation:** $d / D$
- **Thinness:** $2r / D$
- **Roundness:** $4A / (\pi D^2)$
- **Crofton Ratio:** $P_{\text{Crofton}} / (\pi D)$

In [ ]:
def extract_shape_descriptors(images_dir: Path):
    classes = ["apple", "Bone", "camel"]
    elongation, thinness, roundness, crofton_ratio, labels = [], [], [], [], []

    for class_name in classes:
        files = sorted(
            images_dir.glob(f"{class_name}-*.bmp"),
            key=lambda p: int(p.stem.split("-")[-1]),
        )
        print(f"Processing class '{class_name}': {len(files)} images")

        for path in files:
            img = imageio.imread(path)
            if img.ndim == 3:
                img = img[:, :, 0]

            # Ensure object is foreground (True), handling both bool and uint8 images
            mask = img.astype(bool) if img.dtype == bool else (img > 127)
            if np.sum(mask) > mask.size / 2:
                mask = ~mask

            r = inscribed_radius(mask)
            d, D = feret_diameters(mask)
            p = crofton_perimeter(mask)
            area = float(np.sum(mask))

            elongation.append(d / D)
            thinness.append(2.0 * r / D)
            roundness.append(4.0 * area / (np.pi * D**2))
            crofton_ratio.append(p / (np.pi * D))
            labels.append(class_name)

    return (
        np.array(elongation),
        np.array(thinness),
        np.array(roundness),
        np.array(crofton_ratio),
        np.array(labels),
    )


elongation, thinness, roundness, crofton_ratio, labels = extract_shape_descriptors(KIMIA_DIR)
print(f"Extracted descriptors for {len(labels)} shapes.")

## 3. Shape Diagram Evaluation & K-Means Clustering

We evaluate how well each 2D shape diagram separates the three classes using unsupervised K-Means clustering (`k = 3`).

In [ ]:
def evaluate_shape_diagram(x, y, x_label: str, y_label: str) -> float:
    """Cluster 2D shape descriptors with K-Means, plot true vs predicted clusters, and return purity accuracy."""
    pts = np.column_stack((x, y))
    kmeans = KMeans(init="k-means++", n_clusters=3, n_init=10, random_state=42)
    pred = kmeans.fit_predict(pts)
    centers = kmeans.cluster_centers_

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
    colors = ["#4EACC5", "#FF9C34", "#4E9A06"]
    markers = ["o", "^", "s"]
    classes = ["apple", "Bone", "camel"]

    for idx, cls in enumerate(classes):
        mask = labels == cls
        ax1.scatter(x[mask], y[mask], label=cls, color=colors[idx], marker=markers[idx], s=70, alpha=0.85)
    ax1.set_title("Ground-Truth Classes")
    ax1.set_xlabel(x_label)
    ax1.set_ylabel(y_label)
    ax1.legend()

    for k, col in enumerate(colors):
        members = pred == k
        ax2.scatter(pts[members, 0], pts[members, 1], color=col, s=60, alpha=0.75, label=f"Cluster {k}")
        ax2.scatter(centers[k, 0], centers[k, 1], color=col, marker="X", edgecolor="k", s=200)
    ax2.set_title("K-Means Clusters (k = 3)")
    ax2.set_xlabel(x_label)
    ax2.set_ylabel(y_label)
    ax2.legend()

    fig.suptitle(f"Shape Diagram: {x_label} vs {y_label}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # Purity accuracy across the three 20-image blocks
    correct = 0
    for cls in classes:
        cls_preds = pred[labels == cls]
        majority = scipy.stats.mode(cls_preds, keepdims=False).mode
        correct += int(np.sum(cls_preds == majority))
    accuracy = 100.0 * correct / len(labels)
    print(f"Clustering Purity Accuracy: {accuracy:.2f}%")
    return accuracy

### Diagram 1: Elongation ($d/D$) vs. Thinness ($2r/D$)

In [ ]:
acc_1 = evaluate_shape_diagram(elongation, thinness, "Elongation (d / D)", "Thinness (2r / D)")

### Diagram 2: Crofton Ratio ($P_{\text{Crofton}} / \pi D$) vs. Roundness ($4A / \pi D^2$)

In [ ]:
acc_2 = evaluate_shape_diagram(crofton_ratio, roundness, "Crofton Ratio (P / pi D)", "Roundness (4A / pi D^2)")

### Diagram 3: Thinness ($2r/D$) vs. Crofton Ratio ($P_{\text{Crofton}} / \pi D$)

In [ ]:
acc_3 = evaluate_shape_diagram(thinness, crofton_ratio, "Thinness (2r / D)", "Crofton Ratio (P / pi D)")

## 4. Summary Comparison

In [ ]:
summary = [
    ("Elongation vs. Thinness", acc_1),
    ("Crofton Ratio vs. Roundness", acc_2),
    ("Thinness vs. Crofton Ratio", acc_3),
]
for name, acc in summary:
    print(f"  {name:<30s} -> {acc:6.2f}%")